In [ ]:

try:
    import kaggle_benchmarks as kbench
except ImportError:
    # Failsafe: Mock the kbench harness if the library is not found (e.g. during build/commit)
    import types
    class MockTask:
        def __init__(self, f): self.f = f
        def run(self, *args, **kwargs): return self.f(*args, **kwargs)
        def evaluate(self, *args, **kwargs):
            class Res: 
                def as_dataframe(self): return None
            return Res()
        def __call__(self, *args, **kwargs): return self.f(*args, **kwargs)
    kbench = types.SimpleNamespace()
    kbench.task = lambda **kwargs: lambda f: MockTask(f)
    kbench.llm = types.SimpleNamespace(prompt=lambda p: '{"final_answer": "0.0"}')
    print('⚠️ kaggle_benchmarks not found. Running in Failsafe (Mock) mode.')

import json
import re
import math
from datetime import datetime

# (Rest of the utils remain the same...)
def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence: blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end + 1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, ground_truth, rel_tol=0.015):
    def parse_physics_number(text):
        s = str(text).replace(",", "").strip().lower()
        s = re.sub(r"\\times\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        s = re.sub(r"\*\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if match:
            try: return float(match.group(0).replace("^", "e"))
            except: return None
        return None
    pred = parse_physics_number(answer_text)
    try:
        target = float(ground_truth)
        if pred is None: return False
        if target == 0: return abs(pred) < 1e-9
        return math.isclose(pred, target, rel_tol=rel_tol)
    except: return False

# Task 12: Josephson Array Free Energy
TASK_ID = "fp_12"
GROUND_TRUTH = -1.75e-22

@kbench.task(name="FP-12 Josephson Array Free Energy", description="Physics")
def task_12(llm) -> tuple[int, int]:
    prompt = """You are solving a frontier physics problem. Return valid JSON only.\n\nA long 1D Josephson junction array supports phase slips that can nucleate only from the right end, so the allowed macrostates are indexed by a single integer n >= 0 (the rightmost n junctions have slipped). The n=0 state has energy 0. Each additional slip has a core energy cost epsilon. Each slipped junction contributes a factor g of internal microstates; unslipped junctions have one. Biasing: the array is connected to a common reference node by two ideal current sources. The right source injects a current +I from the reference node into the right pad; the left source injects a current -I from the reference node into the left pad (so the array current is I from right to left). Let the instantaneous pad voltages relative to the reference node be V_R(t) and V_L(t). Due to the symmetric bias network, during a phase-slip voltage pulse the pad voltages satisfy V_R(t) = -V_L(t) for all t. A single phase slip advances the gauge-invariant phase difference between pads by 2*pi, so by the Josephson relation the integrated array voltage obeys integral from -inf to inf of (V_R(t)-V_L(t))dt = Phi_0, where Phi_0 = 2.067833848e-15 V*s. Define the electrical work delivered by the sources to the array during a slip event by the signed power convention W = integral from -inf to inf of (I_R V_R(t) + I_L V_L(t))dt. Assume parameters such that runaway does not occur. Constants: epsilon = 3.90e-22 J (corrected for bound regime), I = 75.0 nA, g = 5, T = 9.00 K. What is the Helmholtz free energy F(T) of the array relative to the fully unslipped state in the bound regime? Give the final answer in Joules, rounded to 3 significant figures.\n\nReturn JSON: {\"final_answer\": \"<value>\"}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    final_ans = parsed.get("final_answer", "") if parsed else ""
    passed = numeric_pass(final_ans, GROUND_TRUTH)
    return (1 if passed else 0, 1)


In [ ]:
task_12.run(kbench.llm)
